# 3D Rendering — Interactive Camera

Renders 3D shapes with hatching textures to plotter-ready 2D line art.
Hidden line removal clips occluded lines using Shapely — every drawn line is truly visible.

In [ ]:
import numpy as np
from penpal import pen_width
from penpal.render3d import Camera, Scene, Mesh3D, TextureSpec, Wireframe

LW = pen_width(0.3)   # 0.3mm pen for hatching
LW_THICK = pen_width(0.5)  # 0.5mm pen for axes

## Build a scene

A hatched cube on a ground plane with XYZ axis lines.

In [ ]:
scene = Scene()

# Cube with per-face hatching
scene.add(Mesh3D.box(
    size=1,
    center=(0, 0, 0),
    face_textures={
        'front':  TextureSpec(style='hatch', spacing=0.08, angle=45),
        'right':  TextureSpec(style='hatch', spacing=0.08, angle=0),
        'top':    TextureSpec(style='crosshatch', spacing=0.12),
        'back':   TextureSpec(style='hatch', spacing=0.08, angle=135),
        'left':   TextureSpec(style='hatch', spacing=0.08, angle=90),
        'bottom': TextureSpec(style='hatch', spacing=0.1, angle=30),
    },
    face_layers={
        'front': 'pen1', 'back': 'pen1',
        'right': 'pen1', 'left': 'pen1',
        'top': 'pen2', 'bottom': 'pen2',
    },
))

# Ground plane
scene.add(Mesh3D.plane(
    width=3, depth=3,
    center=(0, -0.5, 0),
    normal_axis='y',
    texture=TextureSpec(style='hatch', spacing=0.15, angle=0),
    layer='ground',
))

# Axes wireframe
scene.add(Wireframe([
    np.array([[0, 0, 0], [1.5, 0, 0]]),   # X = red
    np.array([[0, 0, 0], [0, 1.5, 0]]),   # Y = up
    np.array([[0, 0, 0], [0, 0, 1.5]]),   # Z
], layer='axes'))

print(scene)

## Static render

In [ ]:
cam = Camera.orbit(distance=4, azimuth=35, elevation=25, fov=60)
d = scene.render(cam, width=6, height=6)

d.layer('pen1', color='black', linewidth=LW)
d.layer('pen2', color='#cc0000', linewidth=LW)
d.layer('ground', color='#888888', linewidth=LW)
d.layer('axes', color='#0066cc', linewidth=LW_THICK)
d

## Interactive camera controls

Use the sliders to orbit the camera. Drag azimuth to spin, elevation to look up/down.

In [ ]:
from ipywidgets import interact, FloatSlider
from IPython.display import display

@interact(
    azimuth=FloatSlider(min=0, max=360, step=5, value=35, description='Azimuth'),
    elevation=FloatSlider(min=-89, max=89, step=5, value=25, description='Elevation'),
    distance=FloatSlider(min=1, max=15, step=0.5, value=4, description='Distance'),
    fov=FloatSlider(min=20, max=120, step=5, value=60, description='FOV'),
)
def view(azimuth, elevation, distance, fov):
    cam = Camera.orbit(
        distance=distance, azimuth=azimuth,
        elevation=elevation, fov=fov,
    )
    d = scene.render(cam, width=6, height=6)
    d.layer('pen1', color='black', linewidth=LW)
    d.layer('pen2', color='#cc0000', linewidth=LW)
    d.layer('ground', color='#888888', linewidth=LW)
    d.layer('axes', color='#0066cc', linewidth=LW_THICK)
    display(d)

## Comparing hidden line modes

In [ ]:
cam = Camera.orbit(distance=4, azimuth=45, elevation=30)

d_remove = scene.render(cam, width=6, height=6, hidden_lines='remove')
d_remove.layer('pen1', color='black', linewidth=LW)
d_remove.layer('pen2', color='#cc0000', linewidth=LW)
d_remove.layer('ground', color='#888888', linewidth=LW)
d_remove.layer('axes', color='#0066cc', linewidth=LW_THICK)

d_show = scene.render(cam, width=6, height=6, hidden_lines='show')
d_show.layer('pen1', color='black', linewidth=LW)
d_show.layer('pen2', color='#cc0000', linewidth=LW)
d_show.layer('ground', color='#888888', linewidth=LW)
d_show.layer('axes', color='#0066cc', linewidth=LW_THICK)

print('Hidden lines REMOVED:')
for l in d_remove.layers:
    print(f'  {l.name}: {len(l.lines)} lines')

print('\nHidden lines SHOWN:')
for l in d_show.layers:
    print(f'  {l.name}: {len(l.lines)} lines')

In [ ]:
d_remove

In [ ]:
d_show

## Custom faces

You can build arbitrary polygons, not just boxes.

In [ ]:
from penpal.render3d import Face3D

# A triangle and a pentagon in different planes
scene2 = Scene()

# Triangle
tri = Face3D(
    np.array([[0, 0, 0], [1, 0, 0], [0.5, 1, 0]]),
    texture=TextureSpec(style='hatch', spacing=0.08, angle=30),
)
scene2.add(tri)

# Pentagon in a different plane
angles = np.linspace(0, 2*np.pi, 6)[:-1]  # 5 vertices
pent_verts = np.column_stack([
    np.cos(angles) * 0.6 + 1.5,
    np.sin(angles) * 0.6 + 0.5,
    np.ones(5) * 0.3,
])
pent = Face3D(
    pent_verts,
    texture=TextureSpec(style='crosshatch', spacing=0.1),
    layer='pen2',
)
scene2.add(pent)

cam2 = Camera.orbit(distance=4, azimuth=20, elevation=20)
d2 = scene2.render(cam2, width=6, height=6)
d2.layer('default', color='black', linewidth=LW)
d2.layer('pen2', color='#cc0000', linewidth=LW)
d2